# Film Commission Turin — Industrial Buildings Scraper

The Film Commission Torino Piemonte (FCTP) maintains a public database of filming locations across Turin, categorised by type. I scraped the **industrial buildings** category to collect architectural metadata and images — factories, warehouses, and production facilities that define the city's urban character.

Each building entry has a title, category tags, period, style, condition, and location. Many buildings also have full photo galleries with 20 to 30 images each.

I used Selenium throughout for both the listing page and each detail page, because the image galleries load lazily via JavaScript. An earlier version of this scraper used BeautifulSoup for the detail pages — but this only caught the first thumbnail per building rather than the full gallery. Switching to Selenium and scrolling the full page before extracting images allows the complete gallery to render and be collected.

All outputs are saved in the same folder as this notebook:
- `fctp_industrial_turin.csv`
- `fctp_images/`

## 1. Setup

In [1]:
!pip install selenium webdriver-manager pandas requests -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\elena\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [2]:
import os, re, time, unicodedata
from urllib.parse import urlparse, urljoin
import requests
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# All outputs saved next to this notebook
HERE       = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\INDUSTRIAL BUILDINGS"

BASE_URL   = "https://www.fctp.it/"
LIST_URL   = "https://www.fctp.it/location_list.php?p=TO&t=edifici_industriali"
OUTPUT_CSV = os.path.join(HERE, "fctp_industrial_turin.csv")
IMAGES_DIR = os.path.join(HERE, "fctp_images")

os.makedirs(IMAGES_DIR, exist_ok=True)

print("Output CSV :", OUTPUT_CSV)
print("Images dir :", IMAGES_DIR)

Output CSV : C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\INDUSTRIAL BUILDINGS\fctp_industrial_turin.csv
Images dir : C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\INDUSTRIAL BUILDINGS\fctp_images


## 2. Utility functions

In [3]:
def safe_filename(name: str, maxlen: int = 80) -> str:
    """Convert a building title into a clean filesystem-safe filename."""
    if not name:
        return "untitled"
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
    name = re.sub(r'[\\/*?"<>|:]+', "_", name)
    name = re.sub(r"\s+", "_", name).strip("_")
    return name[:maxlen] or "untitled"


def download_image(session: requests.Session, url: str, filepath: str) -> bool:
    """Download a single image to a given filepath."""
    try:
        r = session.get(url, timeout=30)
        if r.status_code == 200:
            with open(filepath, "wb") as f:
                f.write(r.content)
            return True
    except Exception as e:
        print(f"    Image download failed: {e}")
    return False

## 3. Browser setup

I use `webdriver-manager` to handle ChromeDriver versioning automatically.

In [4]:
def make_driver(headless: bool = True) -> webdriver.Chrome:
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--window-size=1600,1200")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opts
    )
    driver.set_page_load_timeout(60)
    return driver

## 4. Collecting building links

The listing page renders all building cards via JavaScript. I load it in Selenium, wait for the cards to appear, then collect every detail page link.

In [5]:
def get_building_links() -> list:
    driver = make_driver(headless=True)
    links  = []
    try:
        driver.get(LIST_URL)
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.location-item"))
        )
        time.sleep(1)
        cards = driver.find_elements(
            By.CSS_SELECTOR, "div.location-item a[href*='location']"
        )
        for card in cards:
            href = card.get_attribute("href")
            if href and href not in links:
                links.append(href)
    finally:
        driver.quit()
    print(f"Found {len(links)} building pages")
    return links


building_links = get_building_links()

Found 68 building pages


## 5. Scraping building detail pages

For each building I load the detail page in Selenium and scroll down slowly to trigger lazy loading of the full image gallery. Once the page is fully loaded I extract:

- The building title (just the name, not the breadcrumb)
- Metadata fields: category, period, style, condition, location — found in `div.label` / `div.value` pairs
- All image URLs from `<img>` tags and gallery links

Images are downloaded immediately for each building and saved as `buildingname_1.jpg`, `buildingname_2.jpg`, etc.

In [6]:
def scroll_full_page(driver):
    """Scroll down the page in steps to trigger lazy-loaded images."""
    while True:
        driver.execute_script("window.scrollBy(0, 600);")
        time.sleep(0.4)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if driver.execute_script("return window.scrollY + window.innerHeight") >= new_height:
            break


def extract_title(driver) -> str:
    """Extract just the building name, not the full breadcrumb navigation text."""
    try:
        for selector in ["h1.location-title", "h1", "h2"]:
            els = driver.find_elements(By.CSS_SELECTOR, selector)
            for el in els:
                t = el.text.strip()
                if t and len(t) < 120:
                    return t
    except Exception:
        pass
    return ""


def extract_metadata(driver) -> dict:
    """Extract the structured metadata fields from label/value div pairs."""
    fields = {
        "category":                 "",
        "period":                   "",
        "style":                    "",
        "appearance and condition": "",
        "location":                 ""
    }
    try:
        labels = driver.find_elements(By.CSS_SELECTOR, "div.label")
        for lbl in labels:
            key = lbl.text.strip().lower()
            try:
                val_el = lbl.find_element(
                    By.XPATH, "following-sibling::div[@class='value'][1]"
                )
                value = val_el.text.strip()
            except Exception:
                continue
            if "category" in key:
                fields["category"] = value
            elif "period" in key:
                fields["period"] = value
            elif "style" in key:
                fields["style"] = value
            elif "appearance" in key or "condition" in key:
                fields["appearance and condition"] = value
            elif "location" in key:
                fields["location"] = value
    except Exception as e:
        print(f"    Metadata error: {e}")
    return fields


def extract_gallery_urls(driver, page_url: str) -> list:
    """
    Collect all image URLs from the fully rendered page.
    Checks img src, data-src (lazy load), and anchor hrefs.
    Filters out icons and tiny images by skipping anything
    that looks like a UI asset.
    """
    urls = set()
    skip_patterns = ["logo", "icon", "flag", "arrow", "btn", "sprite", "placeholder"]

    for img in driver.find_elements(By.TAG_NAME, "img"):
        for attr in ["src", "data-src", "data-lazy-src"]:
            src = img.get_attribute(attr) or ""
            if src and any(ext in src.lower() for ext in [".jpg", ".jpeg", ".png"]):
                if not any(p in src.lower() for p in skip_patterns):
                    urls.add(urljoin(page_url, src))

    for a in driver.find_elements(By.TAG_NAME, "a"):
        href = a.get_attribute("href") or ""
        if href and any(ext in href.lower() for ext in [".jpg", ".jpeg", ".png"]):
            if not any(p in href.lower() for p in skip_patterns):
                urls.add(urljoin(page_url, href))

    return list(urls)


def scrape_all_buildings(links: list) -> list:
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0"})
    driver  = make_driver(headless=True)
    rows    = []
    total_images = 0

    try:
        for i, url in enumerate(links, 1):
            print(f"[{i}/{len(links)}] {url}")
            try:
                driver.get(url)
                WebDriverWait(driver, 20).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "body"))
                )
                time.sleep(1.5)

                scroll_full_page(driver)
                time.sleep(1)

                title      = extract_title(driver)
                fields     = extract_metadata(driver)
                image_urls = extract_gallery_urls(driver, url)

                print(f"    Title: {title} | Images found: {len(image_urls)}")

                fname_base = safe_filename(title or f"building_{i}")
                saved = 0
                for j, img_url in enumerate(image_urls, 1):
                    ext      = os.path.splitext(urlparse(img_url).path)[1] or ".jpg"
                    filepath = os.path.join(IMAGES_DIR, f"{fname_base}_{j}{ext}")
                    if not os.path.exists(filepath):
                        if download_image(session, img_url, filepath):
                            saved += 1
                    else:
                        saved += 1

                total_images += saved
                print(f"    Saved {saved} images")

                rows.append({
                    "title":      title,
                    "detail_url": url,
                    "images":     saved,
                    **fields
                })

            except Exception as e:
                print(f"    Error: {e}")

            time.sleep(0.5)

    finally:
        driver.quit()

    print(f"\nDone. {len(rows)} buildings | {total_images} total images")
    return rows


rows = scrape_all_buildings(building_links)

[1/68] https://www.fctp.it/location_item.php?id=4314&t=edifici_industriali&e=&s=&a=&p=TO&c=&q=
    Title:  | Images found: 14
    Saved 14 images
[2/68] https://www.fctp.it/location_item.php?id=3974&t=edifici_industriali&e=&s=&a=&p=TO&c=&q=
    Title:  | Images found: 76
    Saved 76 images
[3/68] https://www.fctp.it/location_item.php?id=3788&t=edifici_industriali&e=&s=&a=&p=TO&c=&q=
    Title:  | Images found: 138
    Saved 138 images
[4/68] https://www.fctp.it/location_item.php?id=3786&t=edifici_industriali&e=&s=&a=&p=TO&c=&q=
    Title:  | Images found: 84
    Saved 84 images
[5/68] https://www.fctp.it/location_item.php?id=4045&t=edifici_industriali&e=&s=&a=&p=TO&c=&q=
    Title:  | Images found: 26
    Saved 26 images
[6/68] https://www.fctp.it/location_item.php?id=5090&t=edifici_industriali&e=&s=&a=&p=TO&c=&q=
    Title:  | Images found: 18
    Saved 18 images
[7/68] https://www.fctp.it/location_item.php?id=4649&t=edifici_industriali&e=&s=&a=&p=TO&c=&q=
    Title:  | Images found:

In [11]:
# Metadata
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def scrape_metadata_only(links: list) -> pd.DataFrame:
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0"})
    driver = make_driver(headless=True)
    rows   = []

    try:
        for i, url in enumerate(links, 1):
            print(f"[{i}/{len(links)}] {url}")
            try:
                driver.get(url)
                time.sleep(2)

                # Title
                title = ""
                try:
                    title = driver.find_element(
                        By.CSS_SELECTOR, "div.page-title"
                    ).text.strip()
                except Exception:
                    pass

                # Location info (e.g. "Carmagnola (TO)\nCODICE 4314")
                location_info = ""
                try:
                    location_info = driver.find_element(
                        By.CSS_SELECTOR, "div.info-container"
                    ).text.strip()
                except Exception:
                    pass

                # Metadata — labels are uppercase text, values follow on the right
                # The page uses a table-like layout with text directly in the row
                fields = {
                    "category":                 "",
                    "period":                   "",
                    "style":                    "",
                    "appearance and condition": "",
                    "location":                 ""
                }

                # Get all text from the page and parse label: value pairs
                page_text = driver.find_element(By.CSS_SELECTOR, "body").text

                for line in page_text.split("\n"):
                    line = line.strip()
                    low  = line.lower()
                    if low.startswith("category"):
                        fields["category"] = line.split("\n")[0].replace("CATEGORY", "").replace("Category", "").strip()
                    elif low.startswith("period"):
                        fields["period"] = line.replace("PERIOD", "").replace("Period", "").strip()
                    elif low.startswith("style"):
                        fields["style"] = line.replace("STYLE", "").replace("Style", "").strip()
                    elif low.startswith("appearance"):
                        fields["appearance and condition"] = line.replace("APPEARANCE AND CONDITION", "").replace("Appearance and condition", "").strip()
                    elif low.startswith("location"):
                        fields["location"] = line.replace("LOCATION", "").replace("Location", "").strip()

                # If fields are still empty, try Selenium elements directly
                if not any(fields.values()):
                    try:
                        # Find all elements that could be label+value rows
                        all_els = driver.find_elements(By.CSS_SELECTOR, "td, div, p, span")
                        prev_text = ""
                        for el in all_els:
                            txt = el.text.strip()
                            if not txt:
                                continue
                            low = txt.lower()
                            if low in ["category", "period", "style",
                                       "appearance and condition", "location"]:
                                prev_text = low
                            elif prev_text:
                                if prev_text == "category":
                                    fields["category"] = txt
                                elif prev_text == "period":
                                    fields["period"] = txt
                                elif prev_text == "style":
                                    fields["style"] = txt
                                elif prev_text == "appearance and condition":
                                    fields["appearance and condition"] = txt
                                elif prev_text == "location":
                                    fields["location"] = txt
                                prev_text = ""
                    except Exception:
                        pass

                # Count already saved images for this building
                existing_images = len([
                    f for f in os.listdir(IMAGES_DIR)
                    if f.startswith(safe_filename(title or f"building_{i}"))
                ]) if title else 0

                print(f"    Title: {title}")
                print(f"    Category: {fields['category']} | Period: {fields['period']}")
                print(f"    Location: {fields['location']}")

                rows.append({
                    "title":                    title,
                    "location_info":            location_info,
                    "detail_url":               url,
                    "category":                 fields["category"],
                    "period":                   fields["period"],
                    "style":                    fields["style"],
                    "appearance and condition": fields["appearance and condition"],
                    "location":                 fields["location"],
                })

            except Exception as e:
                print(f"    Error: {e}")

            time.sleep(0.5)

    finally:
        driver.quit()

    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nDone. {len(df)} buildings saved to {OUTPUT_CSV}")
    return df


df_meta = scrape_metadata_only(building_links)
df_meta.head()

[1/68] https://www.fctp.it/location_item.php?id=4314&t=edifici_industriali&e=&s=&a=&p=TO&c=&q=
    Title: Antica Cereria Pettiti
    Category: Urban surroundings, Castles, villas and palazzi, Industrial buildings | Period: 18th Century
    Location: Torino and province
[2/68] https://www.fctp.it/location_item.php?id=3974&t=edifici_industriali&e=&s=&a=&p=TO&c=&q=
    Title: Arca Studios Docks Dora
    Category: Dwellings, Archeology, Commercial buildings, Industrial buildings, Entertainment production | Period: 20th Century - ‘10s and ‘20s, 21th Century
    Location: Torino and province
[3/68] https://www.fctp.it/location_item.php?id=3788&t=edifici_industriali&e=&s=&a=&p=TO&c=&q=
    Title: Area Variante Bunker
    Category: Urban surroundings, Commercial buildings, Industrial buildings, Sports facilities, Monuments, Natural landscape features, Farming | Period: 20th Century - ‘10s and ‘20s, 20th Century - ‘60s
    Location: Torino and province
[4/68] https://www.fctp.it/location_item.p

,title,location_info,detail_url,category,period,style,appearance and condition,location
0,Antica Cereria Pettiti,Carmagnola (TO)\nCODICE 4314,https://www.fctp.it/location_item.php?id=4314&...,"Urban surroundings, Castles, villas and palazz...",18th Century,Eclectic,Traditional,Torino and province
1,Arca Studios Docks Dora,"GUARDA LA MAPPA\nVia Valprato 68, Torino (TO)\...",https://www.fctp.it/location_item.php?id=3974&...,"Dwellings, Archeology, Commercial buildings, I...","20th Century - ‘10s and ‘20s, 21th Century","Contemporary, Liberty, Minimalistic","Alternative, Furnished, Original (not renovate...",Torino and province
2,Area Variante Bunker,"GUARDA LA MAPPA\nVia Niccolò Paganini, 0/200, ...",https://www.fctp.it/location_item.php?id=3788&...,"Urban surroundings, Commercial buildings, Indu...","20th Century - ‘10s and ‘20s, 20th Century - ‘60s",Contemporary,"Alternative, Informal",Torino and province
3,Area Vitali - Parco Dora,"GUARDA LA MAPPA\nParco Dora, Via Borgaro, Tori...",https://www.fctp.it/location_item.php?id=3786&...,"Urban surroundings, Industrial buildings, Spor...",21th Century,Contemporary,-,Torino and province
4,ex Edificio Industriale - Beeozanam,Torino (TO)\nCODICE 4045,https://www.fctp.it/location_item.php?id=4045&...,"Urban surroundings, Archeology, Commercial bui...",20th Century - ‘30s and ‘40s,Designer,"Alternative, Dilapidated, Informal",Torino and province


In [23]:
from collections import Counter
import os, re

# Count images per building index
counts = Counter()
for f in os.listdir(IMAGES_DIR):
    m = re.match(r"building_(\d+)_", f)
    if m:
        counts[int(m.group(1))] += 1

print("Image counts by building index:")
for idx in sorted(counts.keys()):
    print(f"  building_{idx}: {counts[idx]} images")

# Now assign counts to the CSV by row position
df = pd.read_csv(OUTPUT_CSV)

# The scraper used enumerate(links, 1) so index starts at 1
df["images"] = [counts.get(i + 1, 0) for i in range(len(df))]

df.to_csv(OUTPUT_CSV, index=False)
print(f"\nDone. Total images assigned: {df['images'].sum()}")
print(df[["title", "images"]].to_string())

Image counts by building index:
  building_1: 14 images
  building_2: 76 images
  building_3: 138 images
  building_4: 84 images
  building_5: 26 images
  building_6: 18 images
  building_7: 16 images
  building_8: 26 images
  building_9: 36 images
  building_10: 16 images
  building_11: 66 images
  building_12: 8 images
  building_13: 42 images
  building_14: 56 images
  building_15: 18 images
  building_16: 20 images
  building_17: 54 images
  building_18: 22 images
  building_19: 14 images
  building_20: 24 images
  building_21: 42 images
  building_22: 40 images
  building_23: 20 images
  building_24: 20 images
  building_25: 44 images
  building_26: 20 images
  building_27: 8 images
  building_28: 20 images
  building_29: 44 images
  building_30: 44 images
  building_31: 30 images
  building_32: 58 images
  building_33: 32 images
  building_34: 22 images
  building_35: 20 images
  building_36: 12 images
  building_37: 42 images
  building_38: 104 images
  building_39: 20 images
  

In [30]:
import pandas as pd
from collections import Counter
import os, re

# Step 1 — count images by building index from filenames
counts = Counter()
for f in os.listdir(IMAGES_DIR):
    m = re.match(r"building_(\d+)_", f)
    if m:
        counts[int(m.group(1))] += 1

print(f"Total image files found: {sum(counts.values())}")
print(f"Buildings with images: {len(counts)}")

# Step 2 — load CSV fresh
df = pd.read_csv(OUTPUT_CSV)
print(f"CSV rows: {len(df)}")

# Step 3 — assign counts (scraper used enumerate starting at 1)
df["images"] = [counts.get(i + 1, 0) for i in range(len(df))]

# Step 4 — verify before saving
print("\nImage counts assigned:")
print(df[["title", "images"]].to_string())

# Step 5 — save
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"\nSaved to: {OUTPUT_CSV}")

# Step 6 — verify the save worked
df_check = pd.read_csv(OUTPUT_CSV)
print(f"Columns in saved file: {list(df_check.columns)}")
print(f"Images column sum: {df_check['images'].sum()}")

Total image files found: 2254
Buildings with images: 68
CSV rows: 68

Image counts assigned:
                                       title  images
0                     Antica Cereria Pettiti      14
1                    Arca Studios Docks Dora      76
2                       Area Variante Bunker     138
3                   Area Vitali - Parco Dora      84
4        ex Edificio Industriale - Beeozanam      26
5                Birrificio Pian della Mussa      18
6                                     Campus      16
7   Cascina Marchesa - Parco della Pellerina      26
8                 Castello e Parco di Masino      36
9                              Cava di marmo      16
10                     Cavallerizza Caprilli      66
11               Centrale Termica - Olivetti       8
12  Centro dei Servizi Sociali - Ex Olivetti      42
13  Centro ippico - La Collina degli Stivali      56
14                        Centro Polivalente      18
15                            Centro storico      20
16    

## 6. Dataset overview

In [31]:
df = pd.read_csv(OUTPUT_CSV)

print(f"Total buildings  : {len(df)}")
print(f"Total images     : {df['images'].sum()}")
print(f"Avg images/build : {df['images'].mean():.1f}")
print(f"Columns          : {list(df.columns)}")
print()
print("Period breakdown:")
print(df["period"].value_counts())
print()
print("Style breakdown:")
print(df["style"].value_counts())
print()
print("Appearance and condition breakdown:")
print(df["appearance and condition"].value_counts())
print()
df[["title", "period", "style", "location", "images"]].head(10)

Total buildings  : 68
Total images     : 2254
Avg images/build : 33.1
Columns          : ['title', 'location_info', 'detail_url', 'category', 'period', 'style', 'appearance and condition', 'location', 'images']

Period breakdown:
period
20th Century                                                                                                                                                                         8
21th Century                                                                                                                                                                         7
20th Century - ‘50s                                                                                                                                                                  6
20th Century - ‘60s                                                                                                                                                                  6
20th Century - ‘10s and ‘20s   

,title,period,style,location,images
0,Antica Cereria Pettiti,18th Century,Eclectic,Torino and province,14
1,Arca Studios Docks Dora,"20th Century - ‘10s and ‘20s, 21th Century","Contemporary, Liberty, Minimalistic",Torino and province,76
2,Area Variante Bunker,"20th Century - ‘10s and ‘20s, 20th Century - ‘60s",Contemporary,Torino and province,138
3,Area Vitali - Parco Dora,21th Century,Contemporary,Torino and province,84
4,ex Edificio Industriale - Beeozanam,20th Century - ‘30s and ‘40s,Designer,Torino and province,26
5,Birrificio Pian della Mussa,20th Century,"Contemporary, Rustic",Torino and province,18
6,Campus,21th Century,Contemporary,Torino and province,16
7,Cascina Marchesa - Parco della Pellerina,17th Century,"Classical, Rustic",Torino and province,26
8,Castello e Parco di Masino,"Middle ages, 18th Century, 19th Century","Baroque, Classical, Rustic",Torino and province,36
9,Cava di marmo,Middle ages,Medieval,Torino and province,16
